# SSSD-ECG training — balanced sex subset (50% female / 50% male)

Same pipeline as `Train_custum_SSSD_ECG.ipynb`, except the generator train subset is built by:

- Fixed size **N** = same as the original **10% of the PTB-XL train fold** (~1741).
- Target sex ratio **50% female / 50% male**.
- Within each sex, the **joint distribution of the 71 SCP multihot labels** matches the **full train split** (`df_train_real`): one stratum per distinct 71-d label pattern, integer targets via **largest remainder** (same idea as the skewed notebook’s integer strata counts), then **sampling without replacement** per stratum with **`SUBSET_SEED`**.
- The diffusion model is conditioned on a **72-d** vector: **sex** (one channel) concatenated with the **71-d multihot** (as in `Running_unmodified_SSSD_ECG.ipynb`).

Change **`SUBSET_SEED`** to create a second independent subset.

Use a separate session for the skewed notebook to avoid overwriting `ptbxl_train_data.npy` in `sssd/` during training.

In [ ]:
!pip install -q pykeops

In [ ]:
# basic environment + repo setup
import os
import sys
from pathlib import Path

!pip install -q wfdb resampy ishneholterlib pytorch-lightning

REPO_URL = "https://github.com/Anonymous0002396/CMED-Mini-Project.git"
PROJECT_ROOT = Path("/content/CMED-Mini-Project")
REPO_ROOT = PROJECT_ROOT / "SSSD-ECG"

if not PROJECT_ROOT.exists():
 !git clone {REPO_URL} /content/CMED-Mini-Project
else:
 %cd /content/CMED-Mini-Project
 !git pull

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "src" / "ptb_xl"))
sys.path.insert(0, str(REPO_ROOT / "src" / "sssd"))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("REPO_ROOT exists:", REPO_ROOT.exists())
print("SSSD train.py exists:", (REPO_ROOT / "src" / "sssd" / "train.py").exists())
print("SSSD model exists:", (REPO_ROOT / "src" / "sssd" / "models" / "SSSD_ECG.py").exists())

In [ ]:
# mount Drive and extract raw PTB-XL
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

if not os.path.exists('/content/ptbxl.zip'):
 !cp "/content/drive/MyDrive/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.zip" /content/ptbxl.zip

if not os.path.exists('/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1'):
 !unzip -oq /content/ptbxl.zip -d /content/
 print("Extraction complete.")
else:
 print("PTB-XL already extracted, skipping extraction.")

raw_ptbxl = Path("/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1")
print("raw_ptbxl exists:", raw_ptbxl.exists())

In [ ]:
# preprocess PTB-XL at 100 Hz and load processed metadata
import numpy as np
from pathlib import Path

from clinical_ts.ecg_utils import prepare_data_ptb_xl, channel_stoi_default
from clinical_ts.timeseries_utils import reformat_as_memmap, load_dataset

target_fs = 100
data_folder_ptb_xl = Path("/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1")
target_folder_ptb_xl = Path(f"/content/processed_ptb_xl_fs{target_fs}")

if target_folder_ptb_xl.exists():
 !rm -rf {target_folder_ptb_xl}

print("Rebuilding processed PTB-XL at:", target_folder_ptb_xl)

df_ptb_xl, lbl_itos_ptb_xl, mean_ptb_xl, std_ptb_xl = prepare_data_ptb_xl(
 data_path=data_folder_ptb_xl,
 min_cnt=0,
 target_fs=target_fs,
 channels=12,
 channel_stoi=channel_stoi_default,
 target_folder=target_folder_ptb_xl,
 recreate_data=True,
)

reformat_as_memmap(
 df_ptb_xl,
 target_folder_ptb_xl / "memmap.npy",
 data_folder=target_folder_ptb_xl,
 delete_npys=True
)

df_mapped, lbl_itos_dict, mean, std = load_dataset(target_folder_ptb_xl)

print("Processed df shape:", df_mapped.shape)
print("Available label spaces:", list(lbl_itos_dict.keys()))
print("Processed folder:", target_folder_ptb_xl)

In [ ]:
# 71-d multihot labels + sex (+ optional MI diagnostic)
import numpy as np
import pandas as pd

label_key = "label_all"
label_names = np.array(lbl_itos_dict[label_key])
assert len(label_names) == 71, len(label_names)

mi_statement_names = [
 "IMI", "ASMI", "ILMI", "AMI", "ALMI",
 "INJAS", "LMI", "INJAL", "IPLMI", "IPMI",
 "INJIN", "PMI", "INJLA", "INJIL"
]

label_name_to_idx = {str(name): i for i, name in enumerate(label_names)}
mi_label_indices = [label_name_to_idx[name] for name in mi_statement_names]

def multihot_encode(indices, num_classes):
 out = np.zeros(num_classes, dtype=np.float32)
 for i in indices:
 out[i] = 1.0
 return out

def row_multihot_to_binary_mi(row_vec, mi_indices):
 return float(row_vec[mi_indices].sum() > 0)

df_real = df_mapped.copy()

df_real["label_multihot"] = df_real[f"{label_key}_numeric"].apply(
 lambda x: multihot_encode(x, len(label_names))
)

df_real["label_mi"] = df_real["label_multihot"].apply(
 lambda x: row_multihot_to_binary_mi(x, mi_label_indices)
).astype(np.float32)

df_real["sex_binary"] = df_real["sex"].astype(np.float32)

print("label_all size:", len(label_names))
print("Unique sex values:", sorted(df_real["sex_binary"].dropna().unique().tolist()))
print("Overall MI prevalence:", df_real["label_mi"].mean(), int(df_real["label_mi"].sum()), len(df_real))
print("\nCounts by (sex, MI) (diagnostic):")
print(df_real.groupby(["sex_binary", "label_mi"]).size())

In [ ]:
# train/val/test split + N = same count as original 10% stratified draw
max_fold_id = df_real["strat_fold"].max()

df_train_real = df_real[df_real["strat_fold"] < max_fold_id - 1].copy()
df_val_real = df_real[df_real["strat_fold"] == max_fold_id - 1].copy()
df_test_real = df_real[df_real["strat_fold"] == max_fold_id].copy()

print("Full split sizes:")
print("train:", len(df_train_real))
print("val: ", len(df_val_real))
print("test: ", len(df_test_real))

TRAIN_SUBSET_FRAC = 0.10
N_TRAIN_SUBSET = max(1, int(round(TRAIN_SUBSET_FRAC * len(df_train_real))))
print("\nTarget generator/classifier train subset size N:", N_TRAIN_SUBSET, "(10% of train fold)")

In [ ]:
# balanced sex subset — 50% female / 50% male, within-sex joint over 71-d labels = full train
import numpy as np
from collections import defaultdict

SUBSET_SEED = 42
FEMALE_FRACTION = 0.50

rng = np.random.default_rng(SUBSET_SEED)

def _largest_remainder_allocation(n_target, counts_dict):
 """Integer targets proportional to counts_dict, summing exactly to n_target."""
 keys = list(counts_dict.keys())
 if not keys:
 return {}
 total = sum(counts_dict[k] for k in keys)
 if total == 0:
 raise ValueError("empty population for allocation")
 ideal = {k: n_target * (counts_dict[k] / total) for k in keys}
 floors = {k: int(np.floor(ideal[k])) for k in keys}
 rem = n_target - sum(floors.values())
 order = sorted(keys, key=lambda k: ideal[k] - floors[k], reverse=True)
 for i in range(rem):
 floors[order[i]] += 1
 return floors

def _sample_subset_for_sex(df_sex, n_target, rng, label_col="label_multihot"):
 """Strata = unique 71-d multihot among df_sex; sample without replacement per stratum."""
 pools = defaultdict(list)
 for idx, row in df_sex.iterrows():
 mh = np.asarray(row[label_col], dtype=np.float32)
 key = mh.tobytes()
 pools[key].append(idx)
 counts = {k: len(v) for k, v in pools.items()}
 targets = _largest_remainder_allocation(n_target, counts)
 for k in targets:
 targets[k] = min(targets[k], counts[k])
 deficit = n_target - sum(targets.values())
 while deficit > 0:
 spare_pairs = [(counts[k] - targets[k], k) for k in targets if counts[k] > targets[k]]
 if not spare_pairs:
 raise RuntimeError(
 f"Cannot fill sex quota: need {deficit} more samples but no pool capacity left"
 )
 spare_pairs.sort(reverse=True)
 _, k = spare_pairs[0]
 targets[k] += 1
 deficit -= 1
 picked = []
 for k, need in targets.items():
 if need <= 0:
 continue
 pool = np.array(pools[k], dtype=np.int64)
 take = rng.choice(pool, size=need, replace=False)
 picked.append(take)
 return np.sort(np.concatenate(picked))

male_mask = df_train_real["sex_binary"] == 0.0
female_mask = df_train_real["sex_binary"] == 1.0

n_female = int(round(N_TRAIN_SUBSET * FEMALE_FRACTION))
n_male = N_TRAIN_SUBSET - n_female
print("\nTarget counts: n_male =", n_male, ", n_female =", n_female, ", sum =", n_male + n_female)

df_m = df_train_real.loc[male_mask]
df_f = df_train_real.loc[female_mask]
if len(df_m) < n_male or len(df_f) < n_female:
 raise RuntimeError("Not enough male/female rows for requested subset sizes")

print("Unique multihot strata (male / female):", len(df_m), "rows,", len({row.tobytes() for row in df_m['label_multihot']}), "patterns |",
 len(df_f), "rows,", len({row.tobytes() for row in df_f['label_multihot']}), "patterns")

idx_m = _sample_subset_for_sex(df_m, n_male, rng)
idx_f = _sample_subset_for_sex(df_f, n_female, rng)
train_subset_idx = np.sort(np.concatenate([idx_m, idx_f]))
df_train_subset = df_train_real.loc[train_subset_idx].copy().sort_index()

print("\nActual subset size:", len(df_train_subset))
print("Female fraction:", (df_train_subset["sex_binary"] == 1.0).mean())

mh_full_m = np.stack(df_m["label_multihot"].to_numpy())
mh_full_f = np.stack(df_f["label_multihot"].to_numpy())
sub_m = df_train_subset.loc[df_train_subset["sex_binary"] == 0.0, "label_multihot"]
sub_f = df_train_subset.loc[df_train_subset["sex_binary"] == 1.0, "label_multihot"]
mh_sub_m = np.stack(sub_m.to_numpy()) if len(sub_m) else np.zeros((0, mh_full_m.shape[1]))
mh_sub_f = np.stack(sub_f.to_numpy()) if len(sub_f) else np.zeros((0, mh_full_m.shape[1]))

def _mean_per_label(mh):
 return mh.mean(axis=0) if len(mh) else np.zeros(mh_full_m.shape[1])

pm_full_m = _mean_per_label(mh_full_m)
pm_sub_m = _mean_per_label(mh_sub_m)
pm_full_f = _mean_per_label(mh_full_f)
pm_sub_f = _mean_per_label(mh_sub_f)

print("\nWithin-sex mean abs diff of label marginals (subset vs full train):")
print(" male: ", float(np.mean(np.abs(pm_sub_m - pm_full_m))))
print(" female: ", float(np.mean(np.abs(pm_sub_f - pm_full_f))))

print("\nMI | male (subset):", float(df_train_subset[df_train_subset["sex_binary"] == 0.0]["label_mi"].mean()))
print("MI | female (subset):", float(df_train_subset[df_train_subset["sex_binary"] == 1.0]["label_mi"].mean()))
print("\nCounts by (sex, MI) (diagnostic):")
print(df_train_subset.groupby(["sex_binary", "label_mi"]).size())

In [ ]:
# align memmap dataframe and attach sex + 71-d multihot conditioning (72-d total)
import pickle
import numpy as np

with open(target_folder_ptb_xl / "df_memmap.pkl", "rb") as f:
 df_memmap_meta = pickle.load(f)

df_memmap_meta = df_memmap_meta.copy()

df_memmap_meta["sex_binary"] = df_real["sex_binary"].values.astype(np.float32)
df_memmap_meta["label_mi"] = df_real["label_mi"].values.astype(np.float32)
df_memmap_meta["strat_fold"] = df_real["strat_fold"].values

mh = np.stack(df_real["label_multihot"].to_numpy()).astype(np.float32)
sex_col = df_memmap_meta["sex_binary"].to_numpy(dtype=np.float32)[:, None]
cond_matrix = np.concatenate([sex_col, mh], axis=1).astype(np.float32)

df_memmap_meta["cond_sex_scp71"] = list(cond_matrix)

print("df_memmap_meta shape:", df_memmap_meta.shape)
print("cond row shape (expect 72):", np.asarray(cond_matrix[0]).shape)
print("Counts by (sex, MI) (diagnostic):")
print(df_memmap_meta.groupby(["sex_binary", "label_mi"]).size())

In [ ]:
# memmap-backed split dataframes
max_fold_id = df_memmap_meta["strat_fold"].max()

df_val_real_mem = df_memmap_meta[df_memmap_meta["strat_fold"] == max_fold_id - 1].copy()
df_test_real_mem = df_memmap_meta[df_memmap_meta["strat_fold"] == max_fold_id].copy()

df_train_real_mem_subset = df_memmap_meta.loc[df_train_subset.index].copy()

print("Memmap split sizes:")
print("train subset:", len(df_train_real_mem_subset))
print("val: ", len(df_val_real_mem))
print("test: ", len(df_test_real_mem))
print("\nSubset counts by (sex, MI) (diagnostic):")
print(df_train_real_mem_subset.groupby(["sex_binary", "label_mi"]).size())

In [ ]:
# memmap-backed datasets with 72-d condition vectors (sex + 71 SCP multihot)
from clinical_ts.timeseries_utils import TimeseriesDatasetCrops, ToTensor

input_size = 1000
tfms_ptb_xl = ToTensor()

train_real_ds_subset = TimeseriesDatasetCrops(
 df_train_real_mem_subset,
 output_size=input_size,
 data_folder=target_folder_ptb_xl,
 chunk_length=0,
 min_chunk_length=input_size,
 stride=input_size,
 transforms=tfms_ptb_xl,
 annotation=False,
 col_lbl="cond_sex_scp71",
 memmap_filename=target_folder_ptb_xl / "memmap.npy",
)

val_real_ds_full12 = TimeseriesDatasetCrops(
 df_val_real_mem,
 output_size=input_size,
 data_folder=target_folder_ptb_xl,
 chunk_length=0,
 min_chunk_length=input_size,
 stride=input_size,
 transforms=tfms_ptb_xl,
 annotation=False,
 col_lbl="cond_sex_scp71",
 memmap_filename=target_folder_ptb_xl / "memmap.npy",
)

test_real_ds_full12 = TimeseriesDatasetCrops(
 df_test_real_mem,
 output_size=input_size,
 data_folder=target_folder_ptb_xl,
 chunk_length=0,
 min_chunk_length=input_size,
 stride=input_size,
 transforms=tfms_ptb_xl,
 annotation=False,
 col_lbl="cond_sex_scp71",
 memmap_filename=target_folder_ptb_xl / "memmap.npy",
)

sample_x, sample_cond = train_real_ds_subset[0]
print("sample ECG shape:", sample_x.shape)
print("sample cond shape:", getattr(sample_cond, "shape", None), "first 5:", sample_cond[:5] if hasattr(sample_cond, "__getitem__") else sample_cond)
print("train len:", len(train_real_ds_subset))
print("val len:", len(val_real_ds_full12))
print("test len:", len(test_real_ds_full12))

In [ ]:
# materialize train subset into NumPy arrays
train_data_list = []
train_cond_list = []

for i in range(len(train_real_ds_subset)):
 x, cond = train_real_ds_subset[i]
 train_data_list.append(x.numpy().astype(np.float32))
 train_cond_list.append(np.asarray(cond, dtype=np.float32))

train_data = np.stack(train_data_list).astype(np.float32)
train_cond = np.stack(train_cond_list).astype(np.float32)

print("train_data shape:", train_data.shape, train_data.dtype)
print("train_cond shape:", train_cond.shape, train_cond.dtype)

In [ ]:
# save arrays for train.py + persist row indices
from pathlib import Path

sssd_workdir = REPO_ROOT / "src" / "sssd"
sssd_workdir.mkdir(parents=True, exist_ok=True)

np.save(sssd_workdir / "ptbxl_train_data.npy", train_data)
np.save(sssd_workdir / "ptbxl_train_labels.npy", train_cond)

row_indices = df_train_subset.index.to_numpy()
np.save(sssd_workdir / "generator_train_row_indices.npy", row_indices)

print("Saved:", sssd_workdir / "ptbxl_train_data.npy")
print("Saved:", sssd_workdir / "ptbxl_train_labels.npy")
print("Saved row indices:", sssd_workdir / "generator_train_row_indices.npy", "shape", row_indices.shape)

# np.save(sssd_workdir / "ptbxl_train_data_balanced_50f_50m.npy", train_data)
# np.save(sssd_workdir / "ptbxl_train_labels_balanced_50f_50m.npy", train_cond)

DRIVE_SUBSET_DIR = Path("/content/drive/MyDrive/sssd_sex_scp71_balanced_50f_50m_subsets")
DRIVE_SUBSET_DIR.mkdir(parents=True, exist_ok=True)
np.save(DRIVE_SUBSET_DIR / f"generator_train_row_indices_seed{SUBSET_SEED}.npy", row_indices)
np.save(DRIVE_SUBSET_DIR / f"ptbxl_train_data_seed{SUBSET_SEED}.npy", train_data)
np.save(DRIVE_SUBSET_DIR / f"ptbxl_train_labels_seed{SUBSET_SEED}.npy", train_cond)
meta = {
 "subset_seed": int(SUBSET_SEED),
 "female_fraction": FEMALE_FRACTION,
 "N": int(len(row_indices)),
 "conditioning": "sex (1) + label_all multihot (71) = 72",
 "stratification": "per-sex unique 71-d multihot pattern, largest-remainder integer targets, sample wo replacement",
}
import json
with open(DRIVE_SUBSET_DIR / f"subset_meta_seed{SUBSET_SEED}.json", "w") as f:
 json.dump(meta, f, indent=2)
print("Drive: indices + labels meta + ptbxl_train_*.npy copies ->", DRIVE_SUBSET_DIR)

In [ ]:
# JSON config
import json
from pathlib import Path

sssd_workdir = REPO_ROOT / "src" / "sssd"
config_dir = sssd_workdir / "config"
config_dir.mkdir(parents=True, exist_ok=True)

cfg_path = config_dir / "config_SSSD_ECG_sex_scp71_balanced_50f_50m.json"
drive_output_dir = "/content/drive/MyDrive/sssd_sex_scp71_balanced_50f_50m"

sex_scp_cfg = {
 "diffusion_config": {
 "T": 200,
 "beta_0": 0.0001,
 "beta_T": 0.02
 },
 "wavenet_config": {
 "in_channels": 8,
 "out_channels": 8,
 "num_res_layers": 36,
 "res_channels": 256,
 "skip_channels": 256,
 "diffusion_step_embed_dim_in": 128,
 "diffusion_step_embed_dim_mid": 512,
 "diffusion_step_embed_dim_out": 512,
 "s4_lmax": 1000,
 "s4_d_state": 64,
 "s4_dropout": 0.0,
 "s4_bidirectional": 1,
 "s4_layernorm": 1,
 "label_embed_dim": 128,
 "label_embed_classes": 72
 },
 "train_config": {
 "output_directory": drive_output_dir,
 "ckpt_iter": "max",
 "iters_per_ckpt": 2000,
 "iters_per_logging": 1000,
 "n_iters": 21000,
 "learning_rate": 2e-4,
 "batch_size": 8
 },
 "trainset_config": {
 "segment_length": 1000,
 "sampling_rate": 100,
 "finetune_dataset": "ptbxl_sex_scp71_balanced_50f_50m"
 },
 "gen_config": {
 "output_directory": drive_output_dir,
 "ckpt_path": drive_output_dir
 }
}

with open(cfg_path, "w") as f:
 json.dump(sex_scp_cfg, f, indent=2)

print("Wrote:", cfg_path)
print(json.dumps(sex_scp_cfg, indent=2))

Patch `train.py` so `DataLoader` reads `batch_size` from the config.

In [ ]:
from pathlib import Path

train_py = REPO_ROOT / "src" / "sssd" / "train.py"
lines = train_py.read_text().splitlines()
for i, line in enumerate(lines):
 if "trainloader" in line and "DataLoader" in line:
 print(f"LINE {i}: {repr(line)}")

In [ ]:
train_py = REPO_ROOT / "src" / "sssd" / "train.py"
text = train_py.read_text()

if "batch_size=batch_size" in text:
 print("Batch size already patched.")
else:
 if "batch_size=6" not in text:
 raise RuntimeError("Could not find 'batch_size=6' anywhere in train.py")
 text = text.replace("batch_size=6", "batch_size=batch_size")
 train_py.write_text(text)
 print("Patched batch size in:", train_py)

### Train model

Training command for the balanced subset.

In [ ]:
from pathlib import Path

ckpt_dir = Path("/content/drive/MyDrive/sssd_sex_scp71_balanced_50f_50m/ch256_T200_betaT0.02")
print("exists:", ckpt_dir.exists())
if ckpt_dir.exists():
 print(sorted([p.name for p in ckpt_dir.glob("*.pkl")])[-10:])

In [ ]:
import pykeops
print(pykeops.__version__)

In [ ]:
# Train sex + 71-SCP-conditioned SSSD on the balanced subset
%cd /content/CMED-Mini-Project/SSSD-ECG/src/sssd
!python train.py -c config/config_SSSD_ECG_sex_scp71_balanced_50f_50m.json

In [ ]:
#compare some samples of iteration 14k and 20k (for NORM)

In [ ]:
print(3)

In [ ]:
# Healthy ECG generation quality check (14k vs 20k)
# Compare synthetic healthy ECGs from checkpoints 14k and 20k against one real healthy ECG.

import json
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

from models.SSSD_ECG import SSSD_ECG
from utils.util import calc_diffusion_hyperparams, sampling_label
from inference import generate_four_leads
from clinical_ts.timeseries_utils import TimeseriesDatasetCrops, ToTensor

# Go to SSSD directory
%cd /content/CMED-Mini-Project/SSSD-ECG/src/sssd

# Load config used for balanced sex + 71 statements conditioning
cfg_path = Path("config/config_SSSD_ECG_sex_scp71_balanced_50f_50m.json")
cfg = json.loads(cfg_path.read_text())

diffusion_config = cfg["diffusion_config"]
model_config = cfg["wavenet_config"]
train_config = cfg["train_config"]

# Checkpoints to compare
ckpt_dir = Path(train_config["output_directory"]) / "ch256_T200_betaT0.02"
requested_iters = [14000, 20000]
ckpt_paths = {it: ckpt_dir / f"{it}.pkl" for it in requested_iters}

print("Checkpoint dir:", ckpt_dir)
for it, pth in ckpt_paths.items():
 print(f" iter {it}: exists={pth.exists()} ({pth})")

available_iters = [it for it, pth in ckpt_paths.items() if pth.exists()]
if not available_iters:
 raise FileNotFoundError(
 "None of the requested checkpoints (14000, 20000) exist. "
 "Train longer or change requested_iters."
 )
if 20000 not in available_iters:
 print("\n[INFO] 20000.pkl not found. With n_iters=15000, this checkpoint is only available for longer runs.")

# Pick one real healthy ECG from test fold:
# prefer strict NORM-only + non-MI, else fallback to non-MI + NORM-present
label_names_arr = np.array(label_names)
if "NORM" not in set(label_names_arr.tolist()):
 raise ValueError("NORM label not found in label_names")
norm_idx = int(np.where(label_names_arr == "NORM")[0][0])

max_fold_id_local = int(df_memmap_meta["strat_fold"].max())
cond_full = np.stack(df_memmap_meta["cond_sex_scp71"].to_numpy()).astype(np.float32)
labels71 = cond_full[:, 1:] # drop sex channel

is_norm = labels71[:, norm_idx] > 0.5
is_non_mi = df_memmap_meta["label_mi"].to_numpy(dtype=np.float32) == 0.0
is_test = df_memmap_meta["strat_fold"].to_numpy() == max_fold_id_local
is_norm_only = np.isclose(labels71.sum(axis=1), 1.0)

strict_mask = is_test & is_non_mi & is_norm & is_norm_only
fallback_mask = is_test & is_non_mi & is_norm

candidate_idx = np.where(strict_mask)[0]
mode_used = "strict NORM-only"
if len(candidate_idx) == 0:
 candidate_idx = np.where(fallback_mask)[0]
 mode_used = "fallback: non-MI + NORM present"
if len(candidate_idx) == 0:
 raise RuntimeError("No real healthy candidate found in test fold")

pick_idx = int(candidate_idx[0])
real_row = df_memmap_meta.iloc[[pick_idx]].copy()
real_cond = np.asarray(real_row["cond_sex_scp71"].iloc[0], dtype=np.float32)

print("\nSelected real healthy sample mode:", mode_used)
print(
 "Selected index:", int(real_row.index[0]),
 "sex:", float(real_cond[0]),
 "num_active_labels:", int(real_cond[1:].sum())
)

# Load real waveform from memmap
np.string_ = np.bytes_
real_ds = TimeseriesDatasetCrops(
 real_row,
 output_size=1000,
 data_folder=target_folder_ptb_xl,
 chunk_length=0,
 min_chunk_length=1000,
 stride=1000,
 transforms=ToTensor(),
 annotation=False,
 col_lbl="cond_sex_scp71",
 memmap_filename=target_folder_ptb_xl / "memmap.npy",
)
real_x = real_ds[0][0].numpy().astype(np.float32) # (12, 1000)

# Diffusion hyperparams on device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
diffusion_hyperparams = calc_diffusion_hyperparams(**diffusion_config)
for k in diffusion_hyperparams:
 if k != "T":
 diffusion_hyperparams[k] = diffusion_hyperparams[k].to(device)

def generate_one_from_ckpt(ckpt_path: Path, cond_vec: np.ndarray, seed: int = 123):
 torch.manual_seed(seed)
 np.random.seed(seed)

 net = SSSD_ECG(**model_config).to(device)
 checkpoint = torch.load(ckpt_path, map_location="cpu")
 net.load_state_dict(checkpoint["model_state_dict"])
 net.eval()

 cond = torch.tensor(cond_vec[None, :], dtype=torch.float32, device=device)
 with torch.no_grad():
 gen8 = sampling_label(net, (1, 8, 1000), diffusion_hyperparams, cond=cond)
 gen12 = generate_four_leads(gen8)
 return gen12[0].detach().cpu().numpy().astype(np.float32)

# Generate from each available checkpoint (same cond + seed for fairness)
gen_by_iter = {}
for it in available_iters:
 gen_by_iter[it] = generate_one_from_ckpt(ckpt_paths[it], real_cond, seed=123)

# Plot overlays
lead_names = ["I", "II", "V1", "V2", "V3", "V4", "V5", "V6", "III", "aVR", "aVL", "aVF"]
selected_leads = [0, 1, 2, 3, 4, 5] # I, II, V1-V4
t = np.arange(real_x.shape[1]) / 100.0

fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()

for ax, lead_idx in zip(axes, selected_leads):
 ax.plot(t, real_x[lead_idx], label="Real healthy", linewidth=1.2, color="black")
 for it in sorted(gen_by_iter.keys()):
 ax.plot(t, gen_by_iter[it][lead_idx], label=f"Generated {it}", linewidth=1.0, alpha=0.85)
 ax.set_title(f"Lead {lead_names[lead_idx]}")
 ax.set_xlabel("Time (s)")
 ax.set_ylabel("Amplitude")
 ax.grid(alpha=0.2)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=max(2, len(labels)))
fig.suptitle("Healthy ECG comparison: Real vs Generated (14k vs 20k)", y=1.02)
fig.tight_layout()
plt.show()

# Quick numeric summary vs real
for it in sorted(gen_by_iter.keys()):
 g = gen_by_iter[it]
 mse = float(np.mean((g - real_x) ** 2))
 cors = []
 for lead_idx in range(g.shape[0]):
 a = real_x[lead_idx]
 b = g[lead_idx]
 if np.std(a) < 1e-8 or np.std(b) < 1e-8:
 cors.append(np.nan)
 else:
 cors.append(float(np.corrcoef(a, b)[0, 1]))
 mean_corr = float(np.nanmean(cors))
 print(f"iter {it}: mean MSE={mse:.6f}, mean lead corr={mean_corr:.4f}")

print("\nNote: lower MSE / higher correlation helps, but visual morphology and downstream metrics matter most.")